In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os

import specula
specula.init(-1)    # CPU: the notebook only builds data objects, simulations run in a separate process
from specula.mmlib.yaml_overrides import write_yaml_overrides
from save_kalman_data import save_kalman_data
from utils import (calib_dir, result_dir, run_specula, latest_tn, get_noise_propagation,
                   get_aliasing_power, get_atmo_coeffs, get_static_modes)

In [ ]:
# calib_dir, result_dir: imported from utils
overrides_yaml = 'temp_overrides'

In [ ]:
# Filter / simulation parameters: everything below (bias, Kalman data, simulation overrides) is derived from these
dt = 0.0005              # [s] loop time step (main.time_step)
n_subaps, n_modes = 48, 1300
sigma_e = 0.5            # [e-] RON (cred_*.readout_level)
# Total pupil flux per frame of each sensor [e-], as PyrSlopec 'out_flux' reports it
# (the filter rescales the noise with the measured flux anyway, this is only the starting point)
flux = 59600*8.2**2*np.pi/4*400 # flux at V magnitude 7 (magnitude 6.83 900nm, magnitude 8.15 1300nm) 
flux_vis = flux/500        # per 2 ms VIS frame
flux_nir = flux/2000         # per 0.5 ms NIR frame
vis = dict(rMod=3.0, wvl=900,  flux=flux_vis, frame_steps=4, biased=False)   # pyr_vis: reference sensor
nir = dict(rMod=1.0, wvl=1300, flux=flux_nir, frame_steps=1, biased=True)    # pyr_nir: offset estimated

kf_tag = f"kf_ar2_{n_modes}modes_vis{vis['rMod']:1.1f}_nir{nir['rMod']:1.1f}"   # kalman.kalman_filter_data_object
iir_tag = f'plant_inverse_{n_modes}modes'                                          # kpred.iir_filter_data_object
bias_tag = f"nir_bias_{n_modes}modes_vis{vis['rMod']:1.1f}_nir{nir['rMod']:1.1f}"
rec_vis_tag = f"pyr{vis['rMod']:1.1f}_{n_subaps}x{n_subaps}_wvl{vis['wvl']}_bmc2k_vlt_{n_modes}modes_rec"    # pyr_modes_vis.recmat_object
rec_nir_tag = f"pyr{nir['rMod']:1.1f}_{n_subaps}x{n_subaps}_wvl{nir['wvl']}_bmc2k_vlt_{n_modes}modes_rec"    # pyr_modes_nir.recmat_object

In [ ]:
# Static offset of the NIR (biased) sensor w.r.t. the VIS reference, used as initial bias of the Kalman filter.
# Modal measurement (no slope nulls) of the mean residual frame saved by the aliasing calibration (main_simul),
# with the same reconstructors as the closed loop:  bias = rec_nir @ s(frame_nir) - rec_vis @ s(frame_vis)
static_vis = get_static_modes(vis['rMod'], n_subaps, n_modes, vis['wvl'])
static_nir = get_static_modes(nir['rMod'], n_subaps, n_modes, nir['wvl'])
nir_bias = static_nir - static_vis

os.makedirs(os.path.join(calib_dir, 'data'), exist_ok=True)
bias_file = os.path.join(calib_dir, 'data', bias_tag + '.fits')
fits.writeto(bias_file, nir_bias, overwrite=True)
print('saved', bias_file)

x = np.arange(n_modes) + 1
plt.figure()
plt.loglog(x, np.abs(static_vis), label='VIS static')
plt.loglog(x, np.abs(static_nir), label='NIR static')
plt.loglog(x, np.abs(nir_bias), label='NIR bias (NIR - VIS)')
plt.xlabel('mode'); plt.ylabel('|offset| [rec units]')
plt.legend(); plt.grid(which='both', alpha=0.4)

In [ ]:
# Kalman filter data (kalman.kalman_filter_data_object) and plant inverse (kpred.iir_filter_data_object)
a1, a2, q = get_atmo_coeffs(dt)    # AR(2) at the loop rate, more modes than n_modes (needed by aliasing_model='ar2')

sensors = []
for s in (vis, nir):    # same order as kalman.inputs.in_measurements_list
    args = (s['rMod'], n_subaps, n_modes, s['wvl'])
    sensors.append(dict(
        noise_var=get_noise_propagation(*args, sigma_e, s['flux'], pupil_flux=True) ** 2,
        photon_noise_var=get_noise_propagation(*args, 0.0, s['flux'], pupil_flux=True) ** 2,
        reference_flux=s['flux'],
        aliasing=get_aliasing_power(*args, dt=dt),     # single-step (2 kHz) variance -> 'ar2' model
        frame_steps=s['frame_steps'], biased=s['biased']))
sensors[1]['bias'] = fits.getdata(bias_file)        # initial NIR offset (cell above)

fn = save_kalman_data(kf_tag, calib_dir, a1, a2, q, sensors,
                      n_modes=n_modes, aliasing_model='ar2',
                      plant_inverse_tag=iir_tag)
print('saved', fn)

In [ ]:
# Closed-loop simulation with the filter built above (ristretto_main.yml + overrides). Log: specula.log
total_time = 1.0         # [s]
ovdes = ("{"
    f"main.time_step: {dt}, "
    f"main.total_time: {total_time}, "
    f"main.root_dir: '{calib_dir}', "
    f"pyr_vis.pup_diam: {n_subaps}, "
    f"pyr_vis.mod_amp: {vis['rMod']:1.1f}, "
    f"pyr_vis.wavelengthInNm: {vis['wvl']}, "
    f"source_vis.wavelengthInNm: {vis['wvl']}, "
    f"cred_vis.dt: {vis['frame_steps'] * dt}, "
    f"cred_vis.readout_level: {sigma_e}, "
    f"pyr_slopes_vis.pupdata_object: 'pyr_pupdata_{n_subaps}x{n_subaps}', "
    f"pyr_modes_vis.recmat_object: '{rec_vis_tag}', "
    f"pyr_nir.pup_diam: {n_subaps}, "
    f"pyr_nir.mod_amp: {nir['rMod']:1.1f}, "
    f"pyr_nir.wavelengthInNm: {nir['wvl']}, "
    f"source_nir.wavelengthInNm: {nir['wvl']}, "
    f"cred_nir.dt: {nir['frame_steps'] * dt}, "
    f"cred_nir.readout_level: {sigma_e}, "
    f"pyr_slopes_nir.pupdata_object: 'pyr_pupdata_{n_subaps}x{n_subaps}', "
    f"pyr_modes_nir.recmat_object: '{rec_nir_tag}', "
    f"kalman.kalman_filter_data_object: '{kf_tag}', "
    f"kalman.time_step: {dt}, "
    f"kpred.iir_filter_data_object: '{iir_tag}', "
    "}")
write_yaml_overrides(ovdes, temp_name=overrides_yaml)
run_specula('ristretto_main.yml', overrides_yaml, log_file='specula.log')
tn = latest_tn()
print('done:', os.path.join(result_dir, tn))

In [ ]:
# Quick look at the run
sr_vis = fits.getdata(os.path.join(result_dir, tn, 'sr_vis.fits'))
sr_nir = fits.getdata(os.path.join(result_dir, tn, 'sr_nir.fits'))
kf_bias = fits.getdata(os.path.join(result_dir, tn, 'kf_bias.fits'))
print(f'SR VIS: {np.ravel(sr_vis)[-1]:.3f}, SR NIR: {np.ravel(sr_nir)[-1]:.3f}')

plt.figure()
plt.loglog(x, np.abs(nir_bias), label='initial bias')
plt.loglog(x, np.abs(kf_bias[-1][:n_modes]), label='estimated bias (end of run)')
plt.xlabel('mode'); plt.legend(); plt.grid(which='both', alpha=0.4)